# Dogs vs Cats
This notebook runs Part 1: baseline Dogs vs Cats classifiers under identity and fixed tile-wise permutations.


## Setup

### Global / External Imports
Import third-party libraries used only for notebook orchestration and display.


most basic imports

In [ ]:
import sys, os
from pathlib import Path
from typing import Any, Sequence
import socket

### Local Imports
Import project modules. Part 1 orchestration lives in this notebook; reusable training, preprocessing, and evaluation helpers stay in `src`.


prep for local imports

In [ ]:
def get_project_root() -> str:
    project_root = '/Users/royrubin/Documents/GitHub/MLDS_Final_Project'

    if socket.gethostname() != 'MACs-MacBook-Pro.local':
        try:
            from google.colab import drive  # type: ignore # this import is only available in Colab, so if it succeeds we're in Colab
            # mount drive
            drive.mount('/content/drive')
            project_root = '/content/drive/MyDrive/MLDS_Final_Project'
        except Exception as e:
            raise RuntimeError(f"Error mounting Google Drive: {e}")

    return project_root

# Add project root to sys.path so we can import modules from src
REPO_PATH = get_project_root()
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

print(f"Project root: {REPO_PATH}")

make local imports

In [ ]:
# import pandas as pd
# import yaml

In [ ]:
from src.utils.config import CVExperimentConfig
print("Import CVExperimentConfig successful")

In [ ]:
# import matplotlib.pyplot as plt
# print("matplotlib ok")

In [ ]:
# import torch
# print("torch ok")

In [ ]:
# import torchvision
# print("torchvision ok")

In [ ]:
from src.preprocessing.dogs_cats import discover_samples, stratified_split
print("dogs_cats import ok")

In [ ]:
from src.evaluation.experiment_results import (
    build_result_row,
    experiment_output_paths,
    get_device,
    load_experiment_samples,
    plot_accuracy_vs_tiles,
    save_aggregated_accuracy,
    save_rows,
)

In [ ]:
from src.training.experiment_steps import train_model_configuration

In [ ]:
from src.preprocessing.dogs_cats import build_dataloaders, class_counts
from src.preprocessing.permutations import PermutationRecord, build_permutation_records

In [ ]:
from src.utils.io import ensure_dir, save_csv
from src.utils.reproducibility import seed_everything

### Setup configs

In [ ]:
configs = CVExperimentConfig()
display(configs)

### make installations before final external imports

In [ ]:
# Make relevant instalation only if using collab (otherwise, already installed)
from gettext import install

import pip


os.chdir(REPO_PATH)
%pip install -r requirements.txt

if configs.using_google_colab:
    # Install PyTorch with CUDA support in Colab
    %pip uninstall -y torch torchvision
    %pip install --index-url https://download.pytorch.org/whl/cu118 torch torchvision

### final imports (after doing pip install if working on colab)

In [ ]:
import json
import random
from IPython.display import Image, display
import pandas as pd
import numpy as np
import torch
import torchvision

## Experiments

### Experiment helpers
Shared helpers are kept in `src/evaluation/experiment_results.py`; notebook-specific helpers stay here.

In [ ]:
def get_part1_output_paths(*, config: CVExperimentConfig) -> dict[str, str]:
    """Build stable Part 1 output paths for notebook display."""
    paths = experiment_output_paths(
        results_dir=config.results_dir,
        figures_dir=config.figures_dir,
        part_name='part1',
    )
    paths['accuracy_plot'] = paths['figure']
    return paths

### Setup Exp

In [ ]:
# 4. Reproducibility (important for multiple notebooks
random.seed(configs.seed)
np.random.seed(configs.seed)
torch.manual_seed(configs.seed)
torch.set_num_threads(configs.max_threads)  # avoid contention across notebooks

In [ ]:
output_paths = get_part1_output_paths(config=configs)
output_paths

### Data Loading
Discover the configured Dogs vs Cats split before training.


In [ ]:
if not configs.using_google_colab:
    configs.sample_data = True
    configs.sample_limit = 500 # 1000

train_samples, validation_samples, test_samples = load_experiment_samples(config=configs, seed=configs.seed)
print(f'Train samples: {len(train_samples)}')
print(f'Validation samples: {len(validation_samples)}')
print(f'Test samples: {len(test_samples)}')
print('Train class counts:', class_counts(samples=train_samples))
print('Validation class counts:', class_counts(samples=validation_samples))
print('Test class counts:', class_counts(samples=test_samples))

In [ ]:
if configs.plot_samples:
    import matplotlib.pyplot as plt

    cat_samples = [sample for sample in train_samples if sample[1] == 0][:3]
    dog_samples = [sample for sample in train_samples if sample[1] == 1][:3]
    sample_pairs = cat_samples + dog_samples

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for i, (path, label) in enumerate(sample_pairs):
        img = plt.imread(path)
        axes[i].imshow(img)
        axes[i].set_title('Cat' if label == 0 else 'Dog')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

### Experiments - Run Baselines
Run the configured baseline grid/model/permutation sweep. Re-run this cell to regenerate Part 1 outputs.


In [ ]:
# Build and save permutation records
permutation_records = build_permutation_records(
    grid_sizes=configs.grid_sizes,
    num_permutations=configs.num_permutations,
    permutation_seed=configs.permutation_seed,
    include_identity=True,
)
permutation_rows = [record.__dict__ | {'permutation': json.dumps(record.permutation)} for record in permutation_records]
save_csv(data=permutation_rows, path=output_paths['permutations'])
print(f"Saved {len(permutation_records)} permutation records")

In [ ]:
def train_model_records(
    *,
    config: CVExperimentConfig,
    model_name: str,
    run_id: str,
    train_samples: Sequence[tuple[str, int]],
    validation_samples: Sequence[tuple[str, int]],
    permutation_records: Sequence[PermutationRecord],
    seed: int,
    device: torch.device,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    executable_records = [
        record
        for record in permutation_records
        if not (record.grid_size == 1 and record.permutation_id > 0)
    ]
    for record_index, record in enumerate(executable_records, start=1):
        print()
        print("=" * 80)
        print(
            f"[{record_index}/{len(executable_records)}] Running permutation "
            f"grid={record.grid_size}x{record.grid_size}, "
            f"permutation_id={record.permutation_id}, seed={record.permutation_seed}"
        )
        print("Building dataloaders...")
        train_loader, validation_loader = build_dataloaders(
            train_samples=train_samples,
            val_samples=validation_samples,
            image_size=config.image_size,
            grid_size=record.grid_size,
            permutation=record.permutation,
            seed=seed,
            batch_size=config.batch_size,
            num_workers=config.num_workers,
            standard_augmentation=False,
        )
        print(
            "Finished building dataloaders: "
            f"{len(train_loader)} train batches, {len(validation_loader)} validation batches."
        )
        print(f"Training model '{model_name}' on the current permutation...")
        metrics = train_model_configuration(
            config=config,
            model_name=model_name,
            train_loader=train_loader,
            val_loader=validation_loader,
            device=device,
            overrides={'pretrained': config.pretrained and model_name != 'convmixer'},
        )
        print(f"Done training model '{model_name}' on permutation_id={record.permutation_id}.")
        rows.append(
            build_result_row(
                config=config,
                run_id=run_id,
                model_name=model_name,
                record=record,
                seed=seed,
                metrics=metrics,
            )
        )
    return rows

In [ ]:
# Setup reproducibility and load data
seed = configs.seed
seed_everything(seed=seed, deterministic=configs.deterministic)
# Data already loaded above
print(f"Loaded {len(train_samples)} train, {len(validation_samples)} val samples")

In [ ]:
# Prepare shared result accumulation across model runs
all_rows = []
run_id = configs.config_name

### Train ResNet18

In [ ]:
model_name = "resnet18"
device = get_device(config=configs)
rows = train_model_records(
    config=configs,
    model_name=model_name,
    run_id=run_id,
    train_samples=train_samples,
    validation_samples=validation_samples,
    permutation_records=permutation_records,
    seed=seed,
    device=device,
)
all_rows.extend(rows)
save_rows(rows=all_rows, output_path=output_paths['raw_results'])
print(f"Completed training ResNet18 with {len(rows)} runs")

### Train Swin-T

In [ ]:
model_name = "swin_t"
rows = train_model_records(
    config=configs,
    model_name=model_name,
    run_id=run_id,
    train_samples=train_samples,
    validation_samples=validation_samples,
    permutation_records=permutation_records,
    seed=seed,
    device=device,
)
all_rows.extend(rows)
save_rows(rows=all_rows, output_path=output_paths['raw_results'])
print(f"Completed training Swin-T with {len(rows)} runs")

### Train ConvMixer

In [ ]:
model_name = "convmixer"
rows = train_model_records(
    config=configs,
    model_name=model_name,
    run_id=run_id,
    train_samples=train_samples,
    validation_samples=validation_samples,
    permutation_records=permutation_records,
    seed=seed,
    device=device,
)
all_rows.extend(rows)
save_rows(rows=all_rows, output_path=output_paths['raw_results'])
print(f"Completed training ConvMixer with {len(rows)} runs")

In [ ]:
# Aggregate results and plot
raw_results = pd.read_csv(filepath_or_buffer=output_paths['raw_results'])
aggregated_results = save_aggregated_accuracy(
    raw_results=raw_results,
    group_columns=['model_name', 'grid_size', 'num_tiles'],
    output_path=output_paths['aggregated_results'],
)
plot_accuracy_vs_tiles(aggregated=aggregated_results, output_path=output_paths['accuracy_plot'])
aggregated_results

### Experiments - Results Table
Reload the saved aggregated CSV so the report is reproducible from disk.


In [ ]:
saved_results = {
    'raw': pd.read_csv(filepath_or_buffer=output_paths['raw_results']),
    'aggregated': pd.read_csv(filepath_or_buffer=output_paths['aggregated_results']),
}
display(saved_results['aggregated'])

### Experiments - Accuracy Plot
Display the saved accuracy-vs-number-of-tiles plot.


In [ ]:
figure_path = output_paths['accuracy_plot']
if Path(figure_path).exists():
    display(Image(filename=figure_path))
else:
    print(f'Plot not found yet: {figure_path}')
